In [1]:
import sys
sys.path.append('../src/FluoreModel/')

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel

from reslinear_model import ResLinear
from model import GFPRegressionModel

/trinity/home/d_ryabov/.conda/envs/newNucDPosIT/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_WEIGHTS_PATH = "../data/model_weights/regression_model.weights"
ESM_TYPE = "facebook/esm2_t33_650M_UR50D"
BATCH_SIZE = 100
DEVICE = 'cuda'

In [3]:
brightness_model = ResLinear(1280, 10)
brightness_model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, weights_only=True))
brightness_model = brightness_model.eval()

In [6]:
GFPModel = GFPRegressionModel(None, brightness_model, device=DEVICE)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Режим: ESM-2 заморожен


In [7]:
tokenizer = AutoTokenizer.from_pretrained(ESM_TYPE, return_tensors='pt', padding=True, truncation=True)

In [8]:
from pathlib import Path
import glob

# Получить все .fasta файлы
fasta_files = list(Path('../data/mpnn_sequences/MPNN_seqs/').glob('*.fa'))

In [9]:
from  gfp_datasets import create_fasta_dataloader

loader = create_fasta_dataloader(
    fasta_files,      # or a single file / list of files
    batch_size=5,
    num_workers=4,             # parallel prefetch; tune to CPU cores
    shuffle=False,
    dataset_kwargs={"preload": False},
)

In [11]:
import csv
import os

OUTPATH = "../data/scored_sequences/scores.csv"
BUFFER_SIZE = 10000

file_exists = os.path.isfile(OUTPATH)

with open(OUTPATH, 'a' if file_exists else 'w', newline='') as f:
    writer = csv.writer(f)
    
    if not file_exists:
        writer.writerow(['id', 'struct', 'brightness', 'sequence'])
    
    buffer = []
    
    with torch.no_grad():
        for batch in loader:
            sequences = batch['sequences']
            tokens = tokenizer(sequences, return_tensors='pt', padding=True, truncation=True).to(DEVICE)
            predicted_brightness = GFPModel(tokens).cpu().numpy().flatten()
            structs = [record.split(' ')[0] for record in batch['headers']]
            ids = batch['indices'].numpy()
            
            # Добавляем в буфер
            for i in range(len(predicted_brightness)):
                buffer.append([ids[i], structs[i], predicted_brightness[i], sequences[i]])
            
            # Если буфер переполнен - записываем
            if len(buffer) >= BUFFER_SIZE:
                writer.writerows(buffer)
                f.flush()  # Принудительная запись на диск
                buffer.clear()
            break
    
    # Записываем остаток
    if buffer:
        writer.writerows(buffer)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


IndexError: index 5 is out of bounds for axis 0 with size 5

In [12]:
predicted_brightness

array([2.9897618, 2.5585055, 2.6779928, 2.7826004, 2.6649208, 2.9897618,
       2.5585055, 2.6779928, 2.7826004, 2.6649208], dtype=float32)

In [13]:
sequences

['SKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKFICTTGKLPVPWPTLVTTLVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGTYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFKIRHNVEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSVLSKDPNEKRDHMVLLEFVTAAGIAQVQLVESGGALVQPGGSLRLSCAASGFPVNRYSMRWYRQAPGKEREWVAGMSSAGDRSSYEDSVKGRFTISRDDARNTVYLQMNSLKPEDTAVYYCNVNVGFEYWGQGTQVTVSHH',
 'AEGAALLAEPLPVEVRAELDVNGQRAEVRGRGVGDAQRGLLEQVFVCTSGPLPIPWPVLVPTLGQVFARYPEEQRAHDFFRSCLPEGYRQTRRYTFRDDGVLDAEAVVRMEGDTLVTDARITGTGFDPDGPVLGKKIQFTHGDTDVNVTPDRENKGIRMRYTLRLPLEDGGTLEVDVDGVFTPLSDKPVNLPVPHYIRVSVELSRDPGLAADHMVLRQRAVVGGVEAVSLTESGGGKVAPGGSVTLTCKVTGFDVSSHAVSWWRQRPGQPREWVASISADGTTSTVSSALKGRATISRDVAANTVSLQLNNLQPEDTALYFCEVNVGRTVLGQGTELVVEPA',
 'SAGHALLAGPVPVRVEMEFDVNGKKGKIVGEGVGDAQTGTLEQLYTCTTGELPIPWPVLLPTLGQAFTRYPEEMRKHDFFRSCLPEGYYQERVLTFKDDGTLNVKSVVRFEGDTLVVETKIVGTGFDPDGPILGKKLKFEFGDTEVKVKPDKEIKGIKADYTLKLKLEDGGVQNVNVKETYKPISDKPVNIPKEHYIKVSAELSRDPDNPKDHAVVREHAVAGGIEEVSLTMSGGRVVAPGGSVTLTCSVSGVDVSKHEVSWWRQREGQEREWIASISADGKKSEVADEYKGRATISRDVAANK